In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Paths
images_dir = "/content/drive/MyDrive/crack-detection/images"
annotations_dir = "/content/drive/MyDrive/crack-detection/annotations"

# Get filenames without extensions
image_files = {os.path.splitext(f)[0] for f in os.listdir(images_dir) if os.path.isfile(os.path.join(images_dir, f))}
annotation_files = {os.path.splitext(f)[0] for f in os.listdir(annotations_dir) if os.path.isfile(os.path.join(annotations_dir, f))}

# Images without annotations
missing_annotations = image_files - annotation_files
print(f"Number of images missing annotations: {len(missing_annotations)}")
if missing_annotations:
    print("Images missing annotations:")
    for img in sorted(missing_annotations):
        print(img)

# Annotations without images
missing_images = annotation_files - image_files
print(f"\nNumber of annotations missing images: {len(missing_images)}")
if missing_images:
    print("Annotations missing images:")
    for ann in sorted(missing_images):
        print(ann)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of images missing annotations: 0

Number of annotations missing images: 0


In [ ]:
# xml_to_yolo.py
import os
import xml.etree.ElementTree as ET

# Paths (updated to Google Drive)
dataset_dir = '/content/drive/MyDrive/crack-detection'
annotations_dir = os.path.join(dataset_dir, 'annotations')
labels_dir = os.path.join(dataset_dir, 'labels')

os.makedirs(labels_dir, exist_ok=True)

# Define classes (adjust if more types of cracks exist)
classes = ["Longitudinal crack"]  # class index 0

def convert_bbox(size, box):
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    x = (box[0] + box[1]) / 2.0 - 1
    y = (box[2] + box[3]) / 2.0 - 1
    w = box[1] - box[0]
    h = box[3] - box[2]
    x = x * dw
    w = w * dw
    y = y * dh
    h = h * dh
    return (x, y, w, h)

for xml_file in os.listdir(annotations_dir):
    if not xml_file.endswith('.xml'):
        continue
    in_file = os.path.join(annotations_dir, xml_file)
    tree = ET.parse(in_file)
    root = tree.getroot()

    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)

    out_file = os.path.join(labels_dir, os.path.splitext(xml_file)[0] + '.txt')
    with open(out_file, 'w') as f:
        for obj in root.iter('object'):
            cls = obj.find('name').text
            if cls not in classes:
                continue
            cls_id = classes.index(cls)
            xmlbox = obj.find('bndbox')
            b = (
                int(xmlbox.find('xmin').text),
                int(xmlbox.find('xmax').text),
                int(xmlbox.find('ymin').text),
                int(xmlbox.find('ymax').text)
            )
            bb = convert_bbox((w, h), b)
            f.write(f"{cls_id} {' '.join([str(a) for a in bb])}\n")

print("Conversion completed. Labels saved in:", labels_dir)

Conversion completed. Labels saved in: /content/drive/MyDrive/crack-detection/labels


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Paths to your folders in Google Drive
images_dir = '/content/drive/MyDrive/crack-detection/images'
annotations_dir = '/content/drive/MyDrive/crack-detection/annotations'
labels_dir = '/content/drive/MyDrive/crack-detection/labels'

# Function to count files in a directory
def count_files(directory):
    return len([f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))])

# Count files
print(f"Number of image files: {count_files(images_dir)}")
print(f"Number of annotation files: {count_files(annotations_dir)}")
print(f"Number of label files: {count_files(labels_dir)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of image files: 1025
Number of annotation files: 1025
Number of label files: 1025


In [ ]:
import os

labels_dir = "/content/drive/MyDrive/crack-detection/labels"

removed_count = 0

for filename in os.listdir(labels_dir):
    if filename.endswith(".txt"):
        file_path = os.path.join(labels_dir, filename)

        # Check if file is empty (0 bytes)
        if os.path.getsize(file_path) == 0:
            os.remove(file_path)
            removed_count += 1
            print(f"Deleted EMPTY label: {filename}")
            continue

        # Check if file contains only whitespace or blank lines
        with open(file_path, "r") as f:
            content = f.read().strip()

        if content == "":
            os.remove(file_path)
            removed_count += 1
            print(f"Deleted BLANK label: {filename}")

print(f"\nTotal empty labels removed: {removed_count}")

Deleted EMPTY label: 0746_3_4.txt
Deleted EMPTY label: 0745_2_2.txt
Deleted EMPTY label: 0744_4_4.txt
Deleted EMPTY label: 0748_4_4.txt
Deleted EMPTY label: 0745_2_1.txt
Deleted EMPTY label: 0742_3_3.txt
Deleted EMPTY label: 0742_4_4.txt
Deleted EMPTY label: 0743_3_2.txt
Deleted EMPTY label: 0744_1_3.txt
Deleted EMPTY label: 0744_1_2.txt
Deleted EMPTY label: 0743_5_3.txt
Deleted EMPTY label: 0743_2_2.txt
Deleted EMPTY label: 0742_3_4.txt
Deleted EMPTY label: 0743_3_4.txt
Deleted EMPTY label: 0743_5_2.txt
Deleted EMPTY label: 0744_1_1.txt
Deleted EMPTY label: 0744_4_3.txt
Deleted EMPTY label: 0742_4_3.txt
Deleted EMPTY label: 0743_5_1.txt
Deleted EMPTY label: 0743_1_2.txt
Deleted EMPTY label: 0743_4_2.txt
Deleted EMPTY label: 0743_4_3.txt
Deleted EMPTY label: 0744_1_4.txt
Deleted EMPTY label: 0741_3_4.txt
Deleted EMPTY label: 0739_5_5.txt
Deleted EMPTY label: 0739_4_1.txt
Deleted EMPTY label: 0739_5_6.txt
Deleted EMPTY label: 0742_3_2.txt
Deleted EMPTY label: 0741_3_3.txt
Deleted EMPTY 

In [ ]:
import os

# Paths
images_dir = "/content/drive/MyDrive/crack-detection/images"
xml_dir = "/content/drive/MyDrive/crack-detection/annotations"
txt_dir = "/content/drive/MyDrive/crack-detection/labels"

# Collect existing TXT label names (without extension)
txt_files = {os.path.splitext(f)[0] for f in os.listdir(txt_dir)}

deleted_images = 0
deleted_xml = 0

# Check every image in the dataset
for img_file in os.listdir(images_dir):
    if img_file.lower().endswith((".jpg", ".jpeg", ".png")):

        img_name = os.path.splitext(img_file)[0]

        # If image has NO corresponding TXT → delete image + XML
        if img_name not in txt_files:
            img_path = os.path.join(images_dir, img_file)
            xml_path = os.path.join(xml_dir, img_name + ".xml")

            # Delete image
            if os.path.exists(img_path):
                os.remove(img_path)
                deleted_images += 1
                print("Deleted image:", img_file)

            # Delete XML (if exists)
            if os.path.exists(xml_path):
                os.remove(xml_path)
                deleted_xml += 1
                print("Deleted XML:", img_name + ".xml")

print("\nSummary:")
print("Images deleted:", deleted_images)
print("XML files deleted:", deleted_xml)

Deleted image: 0738_5_1.JPG
Deleted XML: 0738_5_1.xml
Deleted image: 0737_4_6.JPG
Deleted XML: 0737_4_6.xml
Deleted image: 0736_1_7.JPG
Deleted XML: 0736_1_7.xml
Deleted image: 0736_5_4.JPG
Deleted XML: 0736_5_4.xml
Deleted image: 0737_2_1.JPG
Deleted XML: 0737_2_1.xml
Deleted image: 0737_4_1.JPG
Deleted XML: 0737_4_1.xml
Deleted image: 0737_2_2.JPG
Deleted XML: 0737_2_2.xml
Deleted image: 0735_4_4.JPG
Deleted XML: 0735_4_4.xml
Deleted image: 0733_1_2.JPG
Deleted XML: 0733_1_2.xml
Deleted image: 0734_1_5.JPG
Deleted XML: 0734_1_5.xml
Deleted image: 0739_2_2.JPG
Deleted XML: 0739_2_2.xml
Deleted image: 0737_0_5.JPG
Deleted XML: 0737_0_5.xml
Deleted image: 0737_1_4.JPG
Deleted XML: 0737_1_4.xml
Deleted image: 0733_1_5.JPG
Deleted XML: 0733_1_5.xml
Deleted image: 0735_1_7.JPG
Deleted XML: 0735_1_7.xml
Deleted image: 0730_4_3.JPG
Deleted XML: 0730_4_3.xml
Deleted image: 0737_2_5.JPG
Deleted XML: 0737_2_5.xml
Deleted image: 0743_4_3.JPG
Deleted XML: 0743_4_3.xml
Deleted image: 0743_5_1.JPG


In [ ]:
# split_dataset_safe.py
import os
import random
import shutil

# Paths (Google Drive)
dataset_dir = '/content/drive/MyDrive/crack-detection'
images_dir = os.path.join(dataset_dir, 'images')
annotations_dir = os.path.join(dataset_dir, 'annotations')
labels_dir = os.path.join(dataset_dir, 'labels')

# Train/val split ratio
split_ratio = 0.8  # 80% train, 20% val

# Check that folders exist
for path in [images_dir, annotations_dir, labels_dir]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Folder not found: {path}")

# List all image files (case-insensitive)
image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Total images found: {len(image_files)}")
print("First 10 images:", image_files[:10])

if len(image_files) == 0:
    raise ValueError("No image files found in the images directory!")

# Shuffle and split
random.shuffle(image_files)
split_index = int(len(image_files) * split_ratio)
train_files = image_files[:split_index]
val_files = image_files[split_index:]

# Create train/val folder structure
for subset in ['train', 'val']:
    for folder_type in ['images', 'annotations', 'labels']:
        os.makedirs(os.path.join(dataset_dir, subset, folder_type), exist_ok=True)

# Function to copy files safely
def copy_files(file_list, subset):
    for file_name in file_list:
        # Copy image
        src_image = os.path.join(images_dir, file_name)
        dst_image = os.path.join(dataset_dir, subset, 'images', file_name)
        shutil.copy2(src_image, dst_image)

        # Copy corresponding annotation (.xml)
        xml_file = os.path.splitext(file_name)[0] + '.xml'
        xml_src = os.path.join(annotations_dir, xml_file)
        xml_dst = os.path.join(dataset_dir, subset, 'annotations', xml_file)
        if os.path.exists(xml_src):
            shutil.copy2(xml_src, xml_dst)
        else:
            print(f"Warning: Annotation not found for {file_name}")

        # Copy corresponding label (.txt)
        txt_file = os.path.splitext(file_name)[0] + '.txt'
        txt_src = os.path.join(labels_dir, txt_file)
        txt_dst = os.path.join(dataset_dir, subset, 'labels', txt_file)
        if os.path.exists(txt_src):
            shutil.copy2(txt_src, txt_dst)
        else:
            print(f"Warning: Label not found for {file_name}")

# Copy files to train/val
print("\nCopying training files...")
copy_files(train_files, 'train')

print("\nCopying validation files...")
copy_files(val_files, 'val')

print("\nDataset split completed!")
print(f"Train: {len(train_files)} images")
print(f"Val: {len(val_files)} images")

# Optional: Verify counts in folders
for subset in ['train', 'val']:
    imgs = len(os.listdir(os.path.join(dataset_dir, subset, 'images')))
    anns = len(os.listdir(os.path.join(dataset_dir, subset, 'annotations')))
    lbls = len(os.listdir(os.path.join(dataset_dir, subset, 'labels')))
    print(f"{subset} folder counts -> Images: {imgs}, Annotations: {anns}, Labels: {lbls}")

Total images found: 1025
First 10 images: ['0738_2_1.JPG', '0738_3_6.JPG', '0745_0_2.JPG', '0740_0_1.JPG', '0742_4_1.JPG', '0740_1_1.JPG', '0743_4_4.JPG', '0741_1_1.JPG', '0747_0_2.JPG', '0741_3_1.JPG']

Copying training files...

Copying validation files...

Dataset split completed!
Train: 820 images
Val: 205 images
train folder counts -> Images: 820, Annotations: 820, Labels: 820
val folder counts -> Images: 205, Annotations: 205, Labels: 205


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Paths to your folders in Google Drive
images_dir = '/content/drive/MyDrive/crack-detection/val/images'
annotations_dir = '/content/drive/MyDrive/crack-detection/val/annotations'
labels_dir = '/content/drive/MyDrive/crack-detection/val/labels'

# Function to count files in a directory
def count_files(directory):
    return len([f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))])

# Count files
print(f"Number of image files: {count_files(images_dir)}")
print(f"Number of annotation files: {count_files(annotations_dir)}")
print(f"Number of label files: {count_files(labels_dir)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of image files: 205
Number of annotation files: 205
Number of label files: 205


In [ ]:
with open("/content/drive/MyDrive/crack-detection/data.yaml", "w") as f:
    f.write("""
train: /content/drive/MyDrive/crack-detection/train/images
val: /content/drive/MyDrive/crack-detection/val/images
nc: 1
names: ["Longitudinal crack"]
""")

In [ ]:
!pip install ultralytics --upgrade

!pip install opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.8 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# Load YOLOv8 small model pretrained on COCO
model = YOLO('yolov8s.pt')

# Train the model
model.train(
    data='/content/drive/MyDrive/crack-detection/data.yaml',  # YAML config
    epochs=50,      # Number of training epochs
    imgsz=512,      # Resize images to 512x512 for training
    batch=8,        # Batch size
    project='/content/drive/MyDrive/crack-detection/crack_detection',  # Save results in Drive
    name='yolov8_crack',
    exist_ok=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.229 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/crack-detection/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x790427822060>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 